# lmm_roi_interaction.ipynb\n\n**Purpose:** Test whether the training effect on Neural Efficiency is\nmoderated by ROI (frontal vs temporal) using a three-way interaction model.\n\n**Inputs:**\n- nirs_neural_efficiency.xlsx — NE index per subject × session × ROI\n\n**Outputs:**\n- LMM results including group × session × ROI interaction\n\n**Model:** NE ~ group * sessao_num * roi + VELmed_baseline + (1|subject) [REML]\n\n**Library:** statsmodels 0.14.6 (Python 3.12)\n\n**Author:** Lucas Gemal (lucasgemal@gmail.com) — IDOR / UFRJ

# LMM — Neural Efficiency | Model Comparison with ROI Interaction
**Project SESI | Input: fnirs_neural_efficiency.xlsx**

Three nested models compared via likelihood ratio test:
- **C1**: `group * sessao_num + roi + VELmed_baseline` (ROI as covariate)
- **C2**: `group * sessao_num + group * roi + sessao_num * roi + VELmed_baseline` (2nd order ROI interactions)
- **C3**: `group * sessao_num * roi + VELmed_baseline` (full triple interaction)

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from scipy.stats import chi2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

## 1 — Configuration

In [ ]:
# Update this path to match your local data directory
PATH_IN        = r'../data/fnirs_neural_efficiency.xlsx'
# Update this path to match your local data directory
PATH_OUT_EXCEL = r'../results/lmm_roi_interaction_results.xlsx'
# Update this path to match your local data directory
PATH_OUT_FIG   = r'../results'

REF_GROUP = 'nao_acelerado'
REF_ROI   = 'FRONTAL'

FORMULAS = {
    'C1': "neural_efficiency ~ group * sessao_num + roi",
    'C2': "neural_efficiency ~ group * sessao_num + group * roi + sessao_num * roi",
    'C3': "neural_efficiency ~ group * sessao_num * roi",
}

## 2 — Load and prepare

In [ ]:
df = pd.read_excel(PATH_IN)

# Set reference categories
df['group'] = pd.Categorical(
    df['group'],
    categories=[REF_GROUP, 'acelerado'],
    ordered=False
)
df['roi'] = pd.Categorical(
    df['roi'],
    categories=[REF_ROI, 'TEMPORAL'],
    ordered=False
)
df['subject'] = pd.Categorical(df['subject'])

# Sanity checks
print(f'Shape          : {df.shape}')
print(f'Subjects       : {df["subject"].nunique()}  (expected 14)')
print(f'Sessions       : {sorted(df["sessao_num"].unique())}')
print(f'ROIs           : {df["roi"].unique().tolist()}')
print(f'Missing NE     : {df["neural_efficiency"].isna().sum()}')
print(f'Missing VELbase: {df["VELmed_baseline"].isna().sum()}')
print(f'\nGroup counts:')
print(df.groupby(['group','roi'], observed=True)['subject'].nunique())

## 3 — Helper functions

In [ ]:
PARAM_LABELS = {
    'group[T.acelerado]'                         : 'Group (Accelerated)',
    'sessao_num'                                 : 'Session Progression',
    'roi[T.TEMPORAL]'                            : 'ROI (Temporal vs Frontal)',
    'VELmed_baseline'                            : 'Baseline Reading Speed',
    'group[T.acelerado]:sessao_num'              : 'Group × Session ★',
    'group[T.acelerado]:roi[T.TEMPORAL]'         : 'Group × ROI',
    'sessao_num:roi[T.TEMPORAL]'                 : 'Session × ROI',
    'group[T.acelerado]:sessao_num:roi[T.TEMPORAL]': 'Group × Session × ROI ★★',
}


def lmm_to_df(res, model_name):
    """Convert LMM fixed effects to clean DataFrame."""
    fe_index = res.fe_params.index
    ci       = res.conf_int().loc[fe_index]
    pvals    = res.pvalues.loc[fe_index]
    bse      = res.bse.loc[fe_index]
    return pd.DataFrame({
        'model'    : model_name,
        'parameter': fe_index,
        'coef'     : res.fe_params.values,
        'se'       : bse.values,
        'z'        : (res.fe_params / bse).values,
        'pvalue'   : pvals.values,
        'ci_lower' : ci[0].values,
        'ci_upper' : ci[1].values,
    }).assign(
        significant=lambda d: d['pvalue'] < 0.05,
        label=lambda d: d['parameter'].map(PARAM_LABELS).fillna(d['parameter'])
    )


def lrt(res_simple, res_complex, df_diff):
    """Likelihood Ratio Test between two nested models."""
    stat = -2 * (res_simple.llf - res_complex.llf)
    p    = chi2.sf(stat, df=df_diff)
    return stat, p


def forest_plot(res, model_name, color, path):
    """Forest plot for a single model."""
    df_res = lmm_to_df(res, model_name)
    df_res = df_res[df_res['parameter'] != 'Intercept'].iloc[::-1].reset_index(drop=True)
    colors = ['crimson' if '★' in str(l) and s else
              color     if s else
              'lightgrey'
              for l, s in zip(df_res['label'], df_res['significant'])]

    fig, ax = plt.subplots(figsize=(13, max(4, len(df_res) * 0.8)))
    for i, row in df_res.iterrows():
        c = colors[i]
        ax.errorbar(
            y=i, x=row['coef'],
            xerr=[[row['coef'] - row['ci_lower']], [row['ci_upper'] - row['coef']]],
            fmt='o', color=c, capsize=5,
            markersize=9, markeredgecolor='black',
            ecolor=c, linewidth=1.5
        )
        p_str = f"p={row['pvalue']:.3f}" if row['pvalue'] >= 0.001 else 'p<0.001'
        ax.text(row['ci_upper'] + 0.01, i, p_str, va='center', fontsize=9,
                color='black' if row['significant'] else 'grey')

    ax.set_yticks(range(len(df_res)))
    ax.set_yticklabels(df_res['label'], fontsize=11)
    ax.axvline(0, linestyle='--', color='grey', linewidth=1.2)
    ax.set_title(f'Model {model_name} — Neural Efficiency LMM',
                 fontsize=13, fontweight='bold', pad=12)
    ax.set_xlabel('Coefficient (β)', fontsize=11)
    ax.grid(axis='x', linestyle='--', alpha=0.5)

    patches = [
        mpatches.Patch(color='crimson',   label='Key term p < 0.05'),
        mpatches.Patch(color=color,       label='p < 0.05'),
        mpatches.Patch(color='lightgrey', label='p ≥ 0.05'),
    ]
    ax.legend(handles=patches, fontsize=9)
    plt.tight_layout()
    plt.savefig(path, dpi=300)
    print(f'  -> Figure saved: {path}')
    plt.show()
    plt.close()

## 4 — Run all three models

In [ ]:
results = {}
colors  = {'C1': 'steelblue', 'C2': 'darkorange', 'C3': 'seagreen'}

for name, formula in FORMULAS.items():
    print(f'\n{"="*60}')
    print(f'MODEL {name}: {formula}')
    print('='*60)
    lmm = smf.mixedlm(formula, data=df, groups=df['subject'])
    res = lmm.fit(reml=False, method='bfgs')
    results[name] = res
    print(res.summary())
    forest_plot(res, name, colors[name], PATH_OUT_FIG + f'forest_{name}.png')

## 5 — Likelihood Ratio Tests

In [ ]:
# C1 vs C2: 2 extra params (group:roi, sessao_num:roi)
stat_12, p_12 = lrt(results['C1'], results['C2'], df_diff=2)

# C2 vs C3: 1 extra param (group:sessao_num:roi)
stat_23, p_23 = lrt(results['C2'], results['C3'], df_diff=1)

# C1 vs C3: 3 extra params
stat_13, p_13 = lrt(results['C1'], results['C3'], df_diff=3)

print('Likelihood Ratio Tests')
print('='*55)
print(f'C1 vs C2  χ²={stat_12:.3f}  df=2  p={p_12:.4f}  {"✅ C2 better" if p_12 < 0.05 else "❌ C1 sufficient"}')
print(f'C2 vs C3  χ²={stat_23:.3f}  df=1  p={p_23:.4f}  {"✅ C3 better" if p_23 < 0.05 else "❌ C2 sufficient"}')
print(f'C1 vs C3  χ²={stat_13:.3f}  df=3  p={p_13:.4f}  {"✅ C3 better" if p_13 < 0.05 else "❌ C1 sufficient"}')
print()

# Model fit summary
print('Model Fit Summary')
print('='*55)
fit_df = pd.DataFrame({
    'model'    : ['C1', 'C2', 'C3'],
    'formula'  : list(FORMULAS.values()),
    'loglik'   : [results[m].llf for m in ['C1','C2','C3']],
    'n_params' : [len(results[m].fe_params) for m in ['C1','C2','C3']],
    'converged': [results[m].converged for m in ['C1','C2','C3']],
})
print(fit_df.to_string(index=False))

## 6 — Key terms comparison across models

In [ ]:
KEY_PARAMS = [
    'group[T.acelerado]:sessao_num',
    'group[T.acelerado]:sessao_num:roi[T.TEMPORAL]',
    'sessao_num:roi[T.TEMPORAL]',
    'VELmed_baseline',
]

all_dfs = pd.concat([lmm_to_df(results[m], m) for m in ['C1','C2','C3']], ignore_index=True)
key_df  = all_dfs[all_dfs['parameter'].isin(KEY_PARAMS)].copy()

print(key_df[['model','label','coef','pvalue','ci_lower','ci_upper','significant']]
      .sort_values(['label','model'])
      .to_string(index=False))

In [ ]:
print(all_dfs.columns.tolist())
print(all_dfs.head(3))

## 7 — Trajectory plot by group and ROI

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

roi_config = [
    ('FRONTAL',  axes[0], 'Frontal ROI',  'steelblue',  'royalblue'),
    ('TEMPORAL', axes[1], 'Temporal ROI', 'darkorange',  'chocolate'),
]

for roi, ax, subtitle, c_acc, c_ctrl in roi_config:
    data = df[df['roi'] == roi]
    traj = (
        data.groupby(['sessao_num', 'group'], observed=True)['neural_efficiency']
        .agg(['mean', 'sem'])
        .reset_index()
    )
    for grp, color, label in [
        ('acelerado',     c_acc,  'Accelerated'),
        ('nao_acelerado', c_ctrl, 'Non-Accelerated')
    ]:
        sub = traj[traj['group'] == grp].sort_values('sessao_num')
        ax.plot(sub['sessao_num'], sub['mean'],
                marker='o', color=color, label=label, linewidth=2)
        ax.fill_between(
            sub['sessao_num'],
            sub['mean'] - sub['sem'],
            sub['mean'] + sub['sem'],
            alpha=0.2, color=color
        )
    ax.axhline(0, linestyle='--', color='grey', linewidth=1)
    ax.set_title(subtitle, fontsize=13, fontweight='bold')
    ax.set_xlabel('Session', fontsize=11)
    ax.set_ylabel('Neural Efficiency (mean ± SEM)', fontsize=11)
    ax.set_xticks(range(1, 10))
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    ax.legend(fontsize=10)

fig.suptitle('Neural Efficiency Trajectory by Group and ROI',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
traj_path = PATH_OUT_FIG + 'trajectory_by_roi.png'
plt.savefig(traj_path, dpi=300, bbox_inches='tight')
print(f'-> Figure saved: {traj_path}')
plt.show()

## 8 — Export all results

In [ ]:
lrt_df = pd.DataFrame({
    'comparison': ['C1 vs C2', 'C2 vs C3', 'C1 vs C3'],
    'chi2'      : [stat_12, stat_23, stat_13],
    'df'        : [2, 1, 3],
    'pvalue'    : [p_12, p_23, p_13],
    'preferred' : [
        'C2' if p_12 < 0.05 else 'C1',
        'C3' if p_23 < 0.05 else 'C2',
        'C3' if p_13 < 0.05 else 'C1',
    ]
})

with pd.ExcelWriter(PATH_OUT_EXCEL, engine='openpyxl') as writer:
    lmm_to_df(results['C1'], 'C1').to_excel(writer, sheet_name='C1_base',           index=False)
    lmm_to_df(results['C2'], 'C2').to_excel(writer, sheet_name='C2_roi_2ndorder',    index=False)
    lmm_to_df(results['C3'], 'C3').to_excel(writer, sheet_name='C3_roi_triple',      index=False)
    all_dfs.to_excel(writer,                         sheet_name='ALL_MODELS',         index=False)
    lrt_df.to_excel(writer,                          sheet_name='LRT_comparison',     index=False)
    fit_df.to_excel(writer,                          sheet_name='model_fit_summary',  index=False)

print(f'\n✅ Results saved: {PATH_OUT_EXCEL}')
print('   Sheets: C1 | C2 | C3 | ALL_MODELS | LRT_comparison | model_fit_summary')